# anybioimage — BioImageViewer Example

This notebook demonstrates the core features of the `anybioimage` package for visualizing biological images in Jupyter.

**Features covered:**
- Loading images with BioImage (TIFF, OME-Zarr)
- Displaying multi-dimensional images with BioImageViewer
- Adding mask overlays
- Using annotation tools (rectangles, polygons, points)
- Fetching annotations as DataFrames — live (reactive) and static
- SAM (Segment Anything Model) integration

## 1. Imports

In [1]:
import ipywidgets as widgets
from IPython.display import display
from bioio import BioImage
import bioio_tifffile
import bioio_ome_zarr
from anybioimage import BioImageViewer

## 2. Load an image

Use `BioImage` to load images in TIFF or OME-Zarr format. Images are represented as 5D arrays: **(T, C, Z, Y, X)**.

In [2]:
# Load from OME-Zarr (lazy loading — efficient for large images)
img = BioImage("../image.zarr", reader=bioio_ome_zarr.Reader)
print("Image shape (TCZYX):", img.shape)

# Alternatively, load from TIFF:
# img = BioImage("../image.tif", reader=bioio_tifffile.Reader)

Image shape (TCZYX): (10, 1, 3, 2048, 2048)


/var/home/maartenpaul/Documents/GitHub/anyimage/.venv/lib/python3.12/site-packages/zarr/core/metadata/v2.py:190: ZarrUserWarning: Found an empty list of filters in the array metadata document. This is contrary to the Zarr V2 specification, and will cause an error in the future. Use None (or Null in a JSON document) instead of an empty list of filters.
  warnings.warn(msg, ZarrUserWarning, stacklevel=1)


## 3. Display the image

Create a `BioImageViewer` widget and pass the `BioImage` object directly. This enables lazy loading and activates T/Z/C sliders when the image has multiple dimensions.

The viewer renders inline in Jupyter — just evaluate the `viewer` variable in a cell.

In [3]:
viewer = BioImageViewer()
viewer.set_image(img)
viewer

## 4. Annotation tools

Use the toolbar in the viewer to draw annotations:

| Tool | Shortcut | Description |
|------|----------|-------------|
| **Pan** | `P` | Navigate and zoom the image |
| **Select** | `V` | Click to select; `Delete` to remove |
| **Rectangle** | `R` | Click and drag to draw bounding boxes |
| **Polygon** | `G` | Click to add vertices; double-click or click near start to close |
| **Point** | `O` | Click to place point markers |

Use the **Layers** dropdown to toggle visibility, adjust opacity, and change mask colors.

Annotations drawn in the viewer automatically sync back to Python — see sections 5 and 6.

## 5. Live annotation output

Jupyter cells are not reactive, but widgets are. Using `observe()` and `ipywidgets.Output()`, you can get a panel that updates **automatically** every time you draw, edit, or delete an annotation in `viewer` above — no cell re-run needed.

Run the cell below once, then draw annotations using the toolbar.

In [4]:
annotation_out = widgets.Output()

def _refresh_annotations(change=None):
    annotation_out.clear_output(wait=True)
    with annotation_out:
        print(f"ROIs: {len(viewer._rois_data)}  "
              f"Polygons: {len(viewer._polygons_data)}  "
              f"Points: {len(viewer._points_data)}")
        print("\n— ROIs —")
        display(viewer.rois_df)
        print("\n— Polygons —")
        display(viewer.polygons_df)
        print("\n— Points —")
        display(viewer.points_df)

viewer.observe(_refresh_annotations, names=['_rois_data', '_polygons_data', '_points_data'])
_refresh_annotations()  # show initial (empty) state
annotation_out

Output()

## 6. Static annotation fetch

Re-run this cell at any time to read the current annotation state directly from Python.

In [10]:
print("=== ROIs ===")
display(viewer.rois_df)

print("\n=== Polygons ===")
display(viewer.polygons_df)

print("\n=== Points ===")
display(viewer.points_df)

=== ROIs ===


,id,x,y,width,height
0,roi_1779485756868,130,116,87,70



=== Polygons ===


,id,points,num_vertices



=== Points ===


,id,x,y


## 7. Add a mask overlay

Overlay segmentation masks on the image. Multiple masks can be added with different names, colors, and opacity.

In [11]:
img_tiff = BioImage("../image.tif", reader=bioio_tifffile.Reader)
mask = BioImage("../mask.tif", reader=bioio_tifffile.Reader)

viewer2 = BioImageViewer()
viewer2.set_image(img_tiff)

mask_id = viewer2.add_mask(
    mask.data,
    name="Segmentation",
    color="#ff0000",
    opacity=0.5,
    contours_only=False
)
print("Added mask with ID:", mask_id)

# Add more masks with different colors:
# viewer2.add_mask(nuclei_mask, name="Nuclei", color="#00ff00", opacity=0.3)
# viewer2.add_mask(cell_mask, name="Cells", color="#0000ff", contours_only=True)

viewer2

Added mask with ID: mask_0_140169820586512


## 8. Mask layer info


In [7]:
print("Active mask IDs:", viewer2.get_mask_ids())

# Update mask settings programmatically:
# viewer2.update_mask_settings(mask_id, opacity=0.3, color="#00ff00")

# Remove a specific mask:
# viewer2.remove_mask(mask_id)

# Clear all masks:
# viewer2.clear_masks()

Active mask IDs: ['mask_0_140177334205776']


## 9. SAM — Segment Anything Model

Enable SAM to automatically generate segmentation masks from rectangle or point annotations.

**How to use:**
1. Select the **Rectangle** tool (`R`) or **Point** tool (`O`)
2. Draw a bounding box or click a point on an object
3. SAM generates a segmentation mask automatically; the ROI/point is consumed as a prompt

**Available models:**
- `mobile_sam` — Fastest, ~40 MB (default)
- `fast_sam` — Fast, CNN-based
- `sam_b` — SAM base model
- `sam_l` — SAM large model

> **Note:** SAM requires PyTorch and the `sam` extra: `uv pip install -e ".[sam]"`

In [8]:
img_sam = BioImage("../image.tif", reader=bioio_tifffile.Reader)

viewer_sam = BioImageViewer()
viewer_sam.set_image(img_sam.data)
viewer_sam.enable_sam(model_type="mobile_sam")
viewer_sam

In [9]:
# Re-run after drawing rectangles or points to inspect the generated SAM masks.
# ROIs and points are consumed as prompts — they won't appear in rois_df.
# The result is a mask layer accessible via get_mask_ids().
print("Mask IDs:", viewer_sam.get_mask_ids())

Mask IDs: []
